# Script

## Objectif: charger le modèle final (Régression Logistique) et prédire si de futurs billets sont vrais ou faux.

In [1]:
import pandas as pd
import json
import joblib

In [2]:
# Charger le pipeline entraîné
model = joblib.load("modele_logreg.joblib")
print("Modèle chargé : modele_logreg.joblib")

# Charger l'ordre des variables attendues
with open("features.json", "r") as f:
    features = json.load(f)
print("Features chargées :", features)

Modèle chargé : modele_logreg.joblib
Features chargées : ['diagonal', 'height_left', 'height_right', 'margin_up', 'length', 'margin_low']


In [3]:
df_new = pd.read_csv("billets_production (1).csv")

# garder uniquement les colonnes attendues dans le bon ordre
df_new = df_new[features]
print("NaN par colonne:\n", df_new[features].isna().sum())

proba_fake = model.predict_proba(df_new)[:, 1]
pred_fake = model.predict(df_new)

resultats = df_new.copy()
resultats["proba_fake"] = proba_fake
resultats["prediction_is_fake"] = pred_fake
resultats["prediction_label"] = resultats["prediction_is_fake"].map({0: "VRAI", 1: "FAUX"})

resultats.to_csv("predictions_billets.csv", index=False)
resultats.head()

NaN par colonne:
 diagonal        0
height_left     0
height_right    0
margin_up       0
length          0
margin_low      0
dtype: int64


,diagonal,height_left,height_right,margin_up,length,margin_low,proba_fake,prediction_is_fake,prediction_label
0,171.76,104.01,103.54,3.30,111.42,5.21,0.999140,1,FAUX
1,171.87,104.17,104.13,3.31,112.09,6.00,0.999881,1,FAUX
2,172.00,104.58,104.29,3.39,111.57,4.99,0.999830,1,FAUX
3,172.49,104.55,104.34,3.03,113.20,4.44,0.036562,0,VRAI
4,171.65,103.63,103.56,3.16,113.33,3.77,0.000260,0,VRAI


In [4]:
# si les noms des variables sont differents par rapport au df initial
# df_new = df_new.rename(columns={"margin_low(mm)": "margin_low"})

In [5]:
# Résumé des prédictions
nb_faux = (resultats["prediction_is_fake"] == 1).sum()
nb_vrai = (resultats["prediction_is_fake"] == 0).sum()
total = len(resultats)

print(f"Résumé : {nb_faux} FAUX / {nb_vrai} VRAI (Total = {total})")

Résumé : 3 FAUX / 2 VRAI (Total = 5)


In [6]:
# Afficher les billets suspectés FAUX
resultats_faux = resultats[resultats["prediction_is_fake"] == 1].copy()
resultats_faux.head()

,diagonal,height_left,height_right,margin_up,length,margin_low,proba_fake,prediction_is_fake,prediction_label
0,171.76,104.01,103.54,3.30,111.42,5.21,0.999140,1,FAUX
1,171.87,104.17,104.13,3.31,112.09,6.00,0.999881,1,FAUX
2,172.00,104.58,104.29,3.39,111.57,4.99,0.999830,1,FAUX


In [7]:
# si volonté d'exporter les faux billets
# resultats_faux.to_csv("billets_suspects_FAUX.csv", index=False)
# print("Export : billets_suspects_FAUX.csv")